In [21]:
import pandas as pd
from pathlib import Path
from urllib.parse import unquote_plus

def load_wiki_plots_dataset(dir_path) -> pd.DataFrame:
    dir_path = Path(dir_path)
    with open(dir_path / 'plots.txt', 'r', encoding='utf-8') as file:
        wiki_plots = file.read().split('<EOS>')
        wiki_plots.pop(-1)
    with open(dir_path / 'exclude_titles.txt', 'r', encoding='utf-8') as file:
        exclude_titles = file.read().splitlines()
    with open(dir_path / 'titles.txt', 'r', encoding='utf-8') as file:
        titles = file.read().splitlines()
    assert len(wiki_plots) == len(titles), f"wiki_plots and titles have different lengths: {len(wiki_plots)} != {len(titles)}"
    
    df = pd.DataFrame({'plot': wiki_plots, 'title': titles})
    df = df[~df['title'].isin(exclude_titles)]
    df["plot"] = df["plot"].apply(lambda x: x.strip())
    df = df[df["plot"].str.len() > 100]
    return df


def load_movie_remakes_dataset(dir):
    filename = "movieRemakesManuallyCleaned.tsv"
    # form readme
    # 1. The dataset is contained in movieRemakesManuallyCleaned.tsv It contains one line for each remake cluster. Each line starts with a cluster id followed by a list of movies that are remakes of each other. A movie is represented using its id, title and summary text.
    with open(Path(dir) / filename, "r", encoding="utf-8") as f:
        lines = f.readlines()

    # each line is a cluster of stories
    # each line starts with a cluster id followed by a list of movies that are remakes of each other. A movie is represented using its id, title and summary text.
    data_points = []
    for line in lines:
        cluster_id, movies = line.split("\t", 1)
        movies = movies.split("\t")
        for start_idx in range(0, len(movies), 3):
            data_points.append({"cluster_id": cluster_id, "title_id": movies[start_idx].strip(), "title": movies[start_idx + 1].strip(), "summary": movies[start_idx + 2].strip()})
    df = pd.DataFrame(data_points)
    return df  


def load_qa_dataset(dir_path):
    dir_path = Path(dir_path)
    df = pd.read_json(dir_path / 'gen_data_google_gemini_2.5_pro_exp_03_25_free.jsonl', lines=True)
    df = df[['question', 'title_id', 'pos_title_ids', 'negative_title_ids', 'aspect']]
    return df


def to_wiki_title(title):
    # Transform from URL-safe format to normal readable format
    # Replace underscores with spaces and decode URL-encoded characters
    return unquote_plus(title.replace("_", " "))


def from_wiki_title(title):
    return title.replace(" ", "_")
    
SAVE_DIR =  Path('D:/DP/data/storysim_dataset')
remakes_dir_path = Path('D:/DP/data/movie_remakes/MovieRemakeDataset_NAACL2018/')
plots_dir_path = Path('D:/DP/data/')
qa_dir_path = Path('D:/DP/data/movie_remakes/MovieRemakeDataset_NAACL2018')

remakes_df = load_movie_remakes_dataset(remakes_dir_path)
plots_df = load_wiki_plots_dataset(plots_dir_path)
qa_df = load_qa_dataset(qa_dir_path)

In [22]:
from sklearn.model_selection import train_test_split 

all_titles = qa_df['title_id'].to_list()
all_titles.extend(qa_df["pos_title_ids"].explode()[qa_df["pos_title_ids"].explode().notna()].tolist())
all_titles.extend(qa_df["negative_title_ids"].explode()[qa_df["negative_title_ids"].explode().notna()].tolist())
all_titles = [to_wiki_title(title) for title in all_titles]
all_titles = set(all_titles)
print(len(all_titles))

plots_df['wiki_title'] = plots_df['title'].apply(to_wiki_title)
remakes_df['wiki_title'] = remakes_df['title'].apply(to_wiki_title)
filtered_plots_df = plots_df[plots_df['wiki_title'].isin(all_titles)]
filtered_remakes_df = remakes_df[remakes_df['wiki_title'].isin(all_titles)] 
wiki_title_to_summary = dict(zip(filtered_plots_df['wiki_title'], filtered_plots_df['plot']))
wiki_title_to_plot = dict(zip(filtered_remakes_df['wiki_title'], filtered_remakes_df['summary']))
title_to_content = {**wiki_title_to_summary, **wiki_title_to_plot}

def get_content(title):
    return title_to_content.get(title, None)

no_content_titles = [title for title in all_titles if not title_to_content.get(title, None)]
print(len(no_content_titles))
no_content_titles_ids = [from_wiki_title(title) for title in no_content_titles]
print("Before cleaning: ", len(qa_df))
qa_df = qa_df[~qa_df['title_id'].isin(no_content_titles_ids)]
print("No content titles: ", len(no_content_titles))
qa_df['pos_title_ids'] = qa_df['pos_title_ids'].apply(lambda x: [to_wiki_title(id) for id in x if id not in no_content_titles_ids])
qa_df['negative_title_ids'] = qa_df['negative_title_ids'].apply(lambda x: [to_wiki_title(id) for id in x if id not in no_content_titles_ids])
qa_df['title_id'] = qa_df['title_id'].apply(to_wiki_title)

# Ensure title_id is included in pos_title_ids for each row
qa_df['pos_title_ids'] = qa_df.apply(lambda row: [row['title_id']] + row['pos_title_ids'] if row['title_id'] not in row['pos_title_ids'] else row['pos_title_ids'], axis=1)

print("After cleaning: ", len(qa_df))
train_idx, test_idx = train_test_split(qa_df.index, test_size=0.3, random_state=42)
qa_df["split"] = "train"
qa_df.loc[test_idx, "split"] = "test"

qa_df.to_csv(SAVE_DIR / "cleaned_qa_dataset.csv", index_label="id") 


2819
809
Before cleaning:  3340
No content titles:  809
After cleaning:  3230


In [23]:
plots_df = plots_df.assign(wiki_title=plots_df['title'].apply(to_wiki_title))
remakes_df = remakes_df.assign(wiki_title=remakes_df['title'].apply(to_wiki_title))

qa_df = qa_df.assign(pos=qa_df['pos_title_ids'].apply(
    lambda titles: [get_content(title) for title in titles]
))

qa_df = qa_df.assign(neg=qa_df['negative_title_ids'].apply(
    lambda titles: [get_content(title) for title in titles]
))

flag_dataset = qa_df[['question', 'pos', 'neg']]
qrels = qa_df[['pos_title_ids', 'negative_title_ids']]

In [24]:
qa_df.head()

,question,title_id,pos_title_ids,negative_title_ids,aspect,split,pos,neg
0,An isolated American research outpost in Antar...,The Thing (1982 film),"[The Thing (1982 film), The Thing from Another...","[The Blob, Invasion of the Body Snatchers (197...",plot,test,[A Norwegian helicopter lands at an American A...,[The film takes place during one long night in...
1,A jury must decide the fate of a young man acc...,12 Angry Men (1957 film),"[12 Angry Men (1957 film), Ek Ruka Hua Faisla,...","[A Few Good Men, Runaway Jury, The Verdict]",plot,train,[The story begins in a courtroom where an 18-y...,[US.\nMarines Lance Corporal Harold Dawson and...
2,A construction worker dissatisfied with his mu...,Total Recall (1990 film),"[Total Recall (1990 film), Total Recall (2012 ...","[Blade Runner, The Matrix, The Thirteenth Floor]",plot,train,"[In 2084, Douglas Quaid is a construction wor...","[In Los Angeles in November 2019, ex-police of..."
4,A former POW from the Korean War experiences r...,The Manchurian Candidate (1962 film),"[The Manchurian Candidate (1962 film), The Man...","[The Parallax View, Three Days of the Condor]",plot,train,"[During the Korean War, the Soviets capture an...",[TV newswoman Lee Carter (Paula Prentiss) is o...
5,Describe a film centered around a charismatic ...,Alfie (1966 film),"[Alfie (1966 film), Alfie (2004 film)]","[American Gigolo, Tom Jones (1963 film), Satur...",character,train,[The film begins with Alfie Elkins ending a r...,[Julian Kaye (Richard Gere) is a male escort i...


In [25]:
save_dir =  Path('D:/DP/data/storysim_dataset')
save_dir.mkdir(parents=True, exist_ok=True)
all_titles = [title for title in all_titles if title_to_content.get(title, None)]
all_plots = [get_content(title) for title in all_titles]

titles_df = pd.DataFrame({"title": list(all_titles), "plot": all_plots})
titles_df.to_csv(save_dir / 'titles.csv', index=False)

flag_dataset.to_csv(save_dir / 'flag_format.csv', index=False)
qrels.to_csv(save_dir / 'qrels.csv', index=False)

In [76]:
from sklearn.model_selection import train_test_split

# Split the dataset into training and testing sets
train_data, test_data = train_test_split(qa_df.index, test_size=0.3, random_state=42)
train_queries = train_data['q'].tolist()
test_queries = test_data['q'].tolist()

titles, passages = zip(*title_to_content.items())

# q ids to pos and neg title indices in the title_to_content dict
titles_list = list(titles)
titles_dict = {title: idx for idx, title in enumerate(titles_list)}

# Use apply with a lambda function for better performance
train_q_to_pos_idx = train_data['pos_wiki_titles'].apply(
    lambda title_list: [titles_dict[title] for title in title_list if title in titles_dict]
).tolist()

test_q_to_pos_idx = test_data['pos_wiki_titles'].apply(
    lambda title_list: [titles_dict[title] for title in title_list if title in titles_dict]
).tolist()


TypeError: remove: path should be string, bytes or os.PathLike, not NoneType

Exception ignored in: 'scipy._lib.messagestream.MessageStream.__dealloc__'
Traceback (most recent call last):
  File "messagestream.pyx", line 91, in scipy._lib.messagestream.MessageStream.close
TypeError: remove: path should be string, bytes or os.PathLike, not NoneType


KeyboardInterrupt: 

In [2]:
from FlagEmbedding import BGEM3FlagModel
from tqdm import tqdm
import numpy as np

model = BGEM3FlagModel(
    model_name_or_path="BAAI/bge-m3",
    device="cuda",
    passage_max_length=4000,
    query_max_length=512,
    
)

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

In [15]:
titles_df = pd.DataFrame({"title": titles_list, "plot": list(title_to_content.values())})
titles_df["length"] = titles_df["plot"].apply(len)
titles_df.sort_values(by="length", inplace=True)
# passages_encoded = []
def get_batch_size(i):
    if i > len(titles_df)/1.5:
        return 2
    elif i > len(titles_df)//2:
        return 4
    elif i > len(titles_df)//3:
        return 16
    elif i > len(titles_df)//6:
        return 32
    else:
        return 64
encoded_passages = []
for i in tqdm(range(0, len(titles_df), 128), desc="Encoding passages"):
    batch_size = get_batch_size(i)
    encoded_passages.append(model.encode(list(titles_df["plot"][i:i+128]), batch_size=batch_size)["dense_vecs"])
   
# passages_encoded = np.concatenate(passages_encoded, axis=0)
# passages_encoded = np.array(passages_encoded)
# np.save("passages_encoded.npy", passages_encoded)


Encoding passages: 100%|██████████| 16/16 [13:20<00:00, 50.04s/it] 


In [17]:
passages_encoded = np.concatenate(encoded_passages, axis=0)
np.save("passages_encoded.npy", passages_encoded)
passages_encoded.shape



(1996, 1024)

In [32]:
titles_df["vector"] = passages_encoded.tolist()
titles_df.sort_index(inplace=True)

In [38]:
train_queries = train_data['q'].tolist()
test_queries = test_data['q'].tolist()

encoded_train_queries = model.encode(train_queries, batch_size=32)["dense_vecs"]
encoded_test_queries = model.encode(test_queries, batch_size=32)["dense_vecs"]


Inference Embeddings: 100%|██████████| 3/3 [00:18<00:00,  6.23s/it]


In [81]:
from nudge import NUDGEN, QueriesAnswersDict


nudge = NUDGEN()
finetuned_embeddings = nudge.finetune_embeddings(
    embeddings=np.array(titles_df["vector"].tolist()),
    train_set=QueriesAnswersDict(
        q_embs=encoded_train_queries.astype(np.double),   
        q_ans_indx=train_q_to_pos_idx
    ),
    val_set=QueriesAnswersDict(
        q_embs=encoded_test_queries.astype(np.double),
        q_ans_indx=test_q_to_pos_idx
    )
)




Calculating G
Finding gamma


In [83]:
import torch
from torch.nn.functional import normalize
import math
import os
def compute_topk_sims_given_shard_path(q_embs, shard_path, k, dist):
    from tqdm import tqdm
    batch_size = 8192

    topks_I =[]
    topks_D =[]
    shard_i = 0
    batch_begin_index = 0
    while os.path.isfile(f"{shard_path}{shard_i}.npy"):
        embeddings_nontrain = np.load(f"{shard_path}{shard_i}.npy")
        shard_i += 1
        for i in tqdm(range(int(math.ceil(len(embeddings_nontrain)/batch_size)))):
            if dist == "cos":
                curr_topk = torch.topk(torch.matmul(q_embs, torch.t(normalize(torch.from_numpy(embeddings_nontrain[i*batch_size:(i+1)*batch_size]).to(q_embs.device)))), k=k, dim=1)
            elif dist == "dot":
                curr_topk = torch.topk(torch.matmul(q_embs, torch.t(torch.from_numpy(embeddings_nontrain[i*batch_size:(i+1)*batch_size]).to(q_embs.device))), k=k, dim=1)
            else:
                assert False, "wrong dist metric"
                
            topks_I.append(curr_topk.indices+batch_begin_index)
            batch_begin_index+=embeddings_nontrain[i*batch_size:(i+1)*batch_size].shape[0]
            topks_D.append(curr_topk.values)
    all_topk = torch.topk(torch.cat(topks_D, dim=1), k=k, dim=1)
    all_I = torch.cat(topks_I, dim=1)
    topk_I = torch.gather(all_I, 1, all_topk.indices)
    topk_D = all_topk.values

    return topk_D, topk_I

def compute_topk_sims_given_embs(q_embs, embeddings_nontrain, k, dist):
    batch_size = 8192

    topks_I =[]
    topks_D =[]
    for i in range(int(math.ceil(len(embeddings_nontrain)/batch_size))):
        if dist == "cos":
            curr_topk = torch.topk(torch.matmul(q_embs, torch.t(normalize(torch.from_numpy(embeddings_nontrain[i*batch_size:(i+1)*batch_size]).to(q_embs.device)))), k=k, dim=1)
        elif dist == "dot":
            curr_topk = torch.topk(torch.matmul(q_embs, torch.t(torch.from_numpy(embeddings_nontrain[i*batch_size:(i+1)*batch_size]).to(q_embs.device))), k=k, dim=1)
        else:
            assert False, "wrong dist metric"
        topks_I.append(curr_topk.indices+i*batch_size)
        topks_D.append(curr_topk.values)
    all_topk = torch.topk(torch.cat(topks_D, dim=1), k=k, dim=1)
    all_I = torch.cat(topks_I, dim=1)
    topk_I = torch.gather(all_I, 1, all_topk.indices)
    topk_D = all_topk.values

    return topk_D, topk_I

def compute_topk_sims(q_embs, nontrain_embeddings_or_shard, k, dist, with_index=False):
    nontrain_embs, path_to_nontrain_emb_shards =  nontrain_embeddings_or_shard
    if nontrain_embs is None and path_to_nontrain_emb_shards is None:
        return None
    if nontrain_embs is not None:
        if nontrain_embs.shape[0] == 0:
            return None
        res = compute_topk_sims_given_embs(q_embs, nontrain_embs, min(k, nontrain_embs.shape[0]), dist)
    else:
        res = compute_topk_sims_given_shard_path(q_embs, path_to_nontrain_emb_shards, k, dist)

    if with_index:
        return res
    return res[0]

class kNNRetriever:
    def __init__(self, embeddings, embeddings_nontrain=None, q_adaptor=None, dist_metric="cos", path_to_embeddings_nontrain_shards=None, device=None):
        assert embeddings_nontrain is None or path_to_embeddings_nontrain_shards is None, "Either read from disk or is in memory"

        if device is None:
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        else:
            self.device = device

        self.embs_nontrain=embeddings_nontrain
        if self.embs_nontrain is not None and self.embs_nontrain.shape[0] == 0:
            self.embs_nontrain=None
        self.path_to_embeddings_nontrain_shards=path_to_embeddings_nontrain_shards
        self.q_adaptor = q_adaptor
        self.dist = dist_metric
        self.embs = embeddings


    def retrieve_topk_from_emb_batch(self, k, q_embeds):
        q_embeds = torch.from_numpy(q_embeds.astype(np.double)).to(self.device)
        if self.q_adaptor is not None:
            q_embeds = self.q_adaptor(q_embeds).detach()

        nontrain_or_path_to_shard = self.embs_nontrain,self.path_to_embeddings_nontrain_shards
        topk_nontrain = compute_topk_sims(q_embeds, nontrain_or_path_to_shard, k, "cos", with_index=True)#always use cosine distne for non-finetuned embeddings

        topk = compute_topk_sims(q_embeds, (self.embs, None), k, self.dist, with_index=True)
        I =  topk[1].cpu().numpy()
        D =  topk[0]

        if topk_nontrain is not None:
            topk_nontrain_D, topk_nontrain_I = topk_nontrain
            preds = np.concatenate([D.cpu().numpy(), topk_nontrain_D.cpu().numpy()], axis=1)
            indxs = np.concatenate([I, topk_nontrain_I.cpu().numpy()+len(self.embs)], axis=1)


            pred_topk_indx = np.argsort(-preds, axis=1)[:, :k]
            I = np.take_along_axis(indxs, pred_topk_indx, axis=1)
            D = np.take_along_axis(preds, pred_topk_indx, axis=1)

        return I 

In [1]:
nudge_n_res = kNNRetriever(finetuned_embeddings).retrieve_topk_from_emb_batch(k=20, q_embeds=encoded_test_queries.astype(np.double))
no_ft_res = kNNRetriever(np.array(titles_df["vector"].tolist())).retrieve_topk_from_emb_batch(k=20, q_embeds=encoded_test_queries.astype(np.double))


def calc_metrics_batch(metrics, top_k_preds, test_anss, test_anss_rel=None):
    all_met_res = np.array([0 for i in range(len(metrics))]).astype(float)
    for i in range(len(top_k_preds)):
        if test_anss_rel is not None:
            curr_test_anss_rel = test_anss_rel[i]
        else:
            curr_test_anss_rel = None

        met_res = calc_metrics(metrics, top_k_preds[i], test_anss[i], curr_test_anss_rel)
        all_met_res += met_res

    return all_met_res/len(top_k_preds)

def calc_metrics(metrics, model_res, true_res, true_res_rel):
    if true_res_rel is None:
        true_res_rel = {}
        if len(true_res) == 0:
            return np.array([0 for i in range(len(metrics))])
        for indx in true_res:
            true_res_rel[indx] = 1
        
    indxs = true_res_rel.keys()
    rels = [true_res_rel[indx] for indx in indxs]
    ordered_rels = np.sort(rels)[::-1]

    met_res = [0 for  i in range(len(metrics))]
    for i, (met_type, k) in enumerate(metrics):
        if met_type == "recall":
            corrects = set()
            for j in range(k):
                pred_indx = model_res[j]
                if pred_indx in true_res:
                    corrects.add(pred_indx)
            met_res[i] = len(corrects)/min(len(true_res) if len(true_res) else 1, k)
        elif met_type == "ndcg":
            ideal_dcg = np.sum([rel/np.log2(loc+2) for loc, rel in enumerate(ordered_rels[:k])])
            rel_scores = np.zeros(k)
            for j in range(k):
                pred_indx = model_res[j]
                if pred_indx in true_res_rel:
                    rel_scores[j] =true_res_rel[pred_indx]

            dcg = np.sum([rel_scores[loc]/np.log2(loc+2) for loc in range(len(rel_scores))])
            met_res[i] = dcg/ideal_dcg
    return np.array(met_res)

metrics = [('recall',20), ('ndcg',20)]
no_ft_accs = calc_metrics_batch(metrics,no_ft_res, test_q_to_pos_idx)
nudgen_accs = calc_metrics_batch(metrics,nudge_n_res, test_q_to_pos_idx)

print(len(no_ft_res))
print(f"No Fine-Tuning {metrics[0][0]}@{metrics[0][1]}: {no_ft_accs[0]*100:.1f}, {metrics[1][0]}@{metrics[1][1]}: {no_ft_accs[1]*100:.1f}")
print(f"NUDGE-N {metrics[0][0]}@{metrics[0][1]}: {nudgen_accs[0]*100:.1f}, {metrics[1][0]}@{metrics[1][1]}: {nudgen_accs[1]*100:.1f}")

NameError: name 'kNNRetriever' is not defined